# Vector Search Concepts

**Module:** 02 — Vector Databases

Similarity search, NN/KNN, exact vs ANN, and hybrid retrieval.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Choose metrics
- Distinguish NN/KNN/ANN/exact
- Design hybrid fusion


## Similarity Search

**Definition.** Find stored vectors maximizing similarity to a query.

**Why it matters.** Core of semantic search/RAG.

**How it works.** Same embedder as index time; top-k under collection metric.

**Intuition.** Drop a pin; nearest pins win.

**Common pitfalls.**
- Embedder mismatch
- Score worship across models

**When to use.** Retrieve by meaning.

| Metric | Use |
|--------|-----|
| Cosine | Text embeddings |
| IP | Normalized dual encoders |
| L2 | Magnitude matters |


In [ ]:
import numpy as np
def cos(a,b): return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+1e-9))
a,b,c=np.array([1.,0]),np.array([2.,0]),np.array([0.,1]); print(cos(a,b),cos(a,c))


In [ ]:
import numpy as np
corpus=['reset password','refunds within 30 days','track shipment']
def emb(t,d=32):
    r=np.random.default_rng(abs(hash(t))%(2**32)); v=r.normal(size=d); return v/(np.linalg.norm(v)+1e-9)
M=np.stack([emb(t) for t in corpus]); s=M@emb('forgot login')
for i in np.argsort(-s): print(round(float(s[i]),3), corpus[i])


In [ ]:
print({'docA':0.82,'docB':0.80}); print('prefer gaps + labels')


### Try it yourself — Similarity Search

1. Cosine vs L2 for normalized embeddings?
2. Test 3 paraphrases.


## Nearest Neighbor (NN)

**Definition.** Single closest vector (1-NN).

**Why it matters.** Dedup/classification building block.

**How it works.** argmax similarity.

**Intuition.** One closest friend.

**Common pitfalls.**
- 1-NN for RAG

**When to use.** Dedup/debug.


In [ ]:
import numpy as np
X=np.array([[1.,0],[.9,.1],[0.,1]]); X/=np.linalg.norm(X,axis=1,keepdims=True)
q=np.array([.95,.05]); q/=np.linalg.norm(q); print(int(np.argmax(X@q)))


In [ ]:
import numpy as np
def dup(E,v,thr=.97):
    b=float((E@v).max()); return b>=thr,b
print(dup(np.eye(2), np.array([.999,.01])/np.linalg.norm([.999,.01])))


In [ ]:
print('1-NN prototype classification: argmax sim to class vectors')


### Try it yourself — Nearest Neighbor (NN)

1. Tune near-dup threshold on 20 pairs.


## KNN

**Definition.** Return k closest vectors.

**Why it matters.** RAG needs multiple candidates.

**How it works.** argpartition + sort; ANN approximates.

**Intuition.** k friends, then rerank.

**Common pitfalls.**
- k too small/large
- Duplicate chunks

**When to use.** Default retrieval mode.


In [ ]:
import numpy as np
rng=np.random.default_rng(0); X=rng.normal(size=(1000,64)); X/=np.linalg.norm(X,axis=1,keepdims=True)+1e-9
q=rng.normal(size=64); q/=np.linalg.norm(q)+1e-9; s=X@q; k=5; i=np.argpartition(-s,k)[:k]; print(i[np.argsort(-s[i])].tolist())


In [ ]:
print({'retrieve_k':50,'rerank_k':8,'context_k':5})


In [ ]:
import numpy as np
def mmr(X,q,k=5,lam=.7):
    s=X@q; sel=[]; cand=set(range(len(X)))
    while len(sel)<k and cand:
        i=max(cand, key=(lambda j: s[j]) if not sel else (lambda j: lam*s[j]-(1-lam)*max(float(X[j]@X[t]) for t in sel)))
        sel.append(i); cand.remove(i)
    return sel
rng=np.random.default_rng(1); X=rng.normal(size=(40,16)); X/=np.linalg.norm(X,axis=1,keepdims=True)+1e-9
print(mmr(X,X[0]))


### Try it yourself — KNN

1. Propose k_retrieve vs k_context.


## ANN — Approximate Nearest Neighbor

**Definition.** Usually-correct neighbors, much faster.

**Why it matters.** Exact won't meet latency at huge N.

**How it works.** HNSW/IVF visit a fraction; knobs move recall–latency.

**Intuition.** Ask a local for directions.

**Common pitfalls.**
- Unchecked recall regressions

**When to use.** Production default beyond small corpora.

```mermaid
flowchart LR
 Q-->A[ANN]-->C[Candidates]-->R[Rescore]-->K[Top-k]
```


In [ ]:
import numpy as np
rng=np.random.default_rng(0); n,k=3000,10; X=rng.normal(size=(n,32)); X/=np.linalg.norm(X,axis=1,keepdims=True)+1e-9
q=X[42]; gold=set(np.argsort(-(X@q))[:k].tolist())
for c in [20,50,200,1000]:
    rec=np.mean([len(gold&set(rng.choice(n,c,False)[np.argsort(-(X[rng.choice(n,c,False)]@q))[:k]].tolist()))/k for _ in range(2)])
    print(c, round(float(rec),2))


In [ ]:
print([('HNSW','efSearch'),('IVF','nprobe')])


In [ ]:
def ok(r,p,rmin=.97,tmax=40): return r>=rmin and p<=tmax
print(ok(.98,35), ok(.99,80))


### Try it yourself — ANN — Approximate Nearest Neighbor

1. Table recall vs latency budget.


## Exact Search

**Definition.** Guarantees true top-k.

**Why it matters.** Oracles, tiny data, rescoring.

**How it works.** Brute-force scan/matmul.

**Intuition.** Check every locker.

**Common pitfalls.**
- Exact online at huge N

**When to use.** Eval + rescoring.


In [ ]:
import numpy as np
def exact(X,q,k=5):
    s=X@q; i=np.argsort(-s)[:k]; return i,s[i]
rng=np.random.default_rng(0); X=rng.normal(size=(200,16)); X/=np.linalg.norm(X,axis=1,keepdims=True)+1e-9
print(exact(X,X[0])[0])


In [ ]:
import numpy as np
rng=np.random.default_rng(1); X=rng.normal(size=(1000,32)); X/=np.linalg.norm(X,axis=1,keepdims=True)+1e-9
q=rng.normal(size=32); q/=np.linalg.norm(q)+1e-9; cand=rng.choice(1000,50,False)
print(cand[np.argsort(-(X[cand]@q))[:5]])


In [ ]:
print([(n, n<20000) for n in [500,5000,500000]])


### Try it yourself — Exact Search

1. Oracle recall@10 for an ANN method.


## Hybrid Search

**Definition.** Blend BM25/sparse with dense vectors.

**Why it matters.** Entities vs paraphrases need both.

**How it works.** Parallel retrieve + RRF/weighted fusion.

**Intuition.** Plates + faces.

**Common pitfalls.**
- Uncalibrated score sums

**When to use.** Enterprise/technical corpora.

```mermaid
flowchart TD
 Q-->B[BM25]; Q-->V[Vector]; B-->F[RRF]; V-->F; F-->O[Hybrid]
```


In [ ]:
from collections import Counter
import numpy as np, math
docs=['error E1234 disk full','refund within 30 days','disk cleanup tips','return policy']
def tok(s): return s.lower().split()
def bm25(q,docs):
    T=[tok(d) for d in docs]; N=len(docs); avg=np.mean([len(t) for t in T]); df=Counter(t for ts in T for t in set(ts)); out=[]
    for toks in T:
        tf=Counter(toks); dl=len(toks); s=0.0
        for term in tok(q):
            if term not in df: continue
            idf=math.log(1+(N-df[term]+.5)/(df[term]+.5)); s+=idf*(tf[term]*2.5)/(tf[term]+1.5*(.25+.75*dl/avg)+1e-9)
        out.append(s)
    return np.array(out)
def emb(t,d=32):
    r=np.random.default_rng(abs(hash(t))%(2**32)); v=r.normal(size=d); return v/(np.linalg.norm(v)+1e-9)
def rrf(lists,k=60):
    sc={}
    for lst in lists:
        for r,i in enumerate(lst): sc[i]=sc.get(i,0)+1/(k+r+1)
    return sorted(sc, key=lambda i:-sc[i])
q='disk error E1234'; br=list(np.argsort(-bm25(q,docs))); dr=list(np.argsort(-(np.stack([emb(d) for d in docs])@emb(q))))
print('rrf', rrf([br,dr]));


In [ ]:
import json
print(json.dumps({'query':'E1234','vector':[0.01,0.2],'alpha':0.5,'top_k':10},indent=2))


In [ ]:
gold={'d0'}
for name,r in {'lex':['d0','d2'],'vec':['d1','d0'],'hyb':['d0','d1']}.items():
    print(name, r[0] in gold)


### Try it yourself — Hybrid Search

1. Implement RRF on your docs.


## Glossary

- **RRF**: Reciprocal Rank Fusion
- **recall@k**: Relevant fraction in top-k


## Summary & Key Takeaways

- Keep metric+embedder+k consistent.
- ANN scales KNN; exact is oracle.
- Hybrid fixes entity blind spots.

### Practice

Write retrieval design: metric, k, ANN, hybrid, eval.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
